# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. We follow best practices by referencing all dataset elements by their schema `@id`, ensuring reproducibility and transparency for FAIR data workflows.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access as a Python object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
# Print main dataset @id and license
print(f"Dataset @id: {metadata.id}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Explore record sets, fields and their `@id`s. All data processing is done by referencing Croissant schema entities via their `@id` to maintain clarity and reproducibility.

Let's inspect the available record sets and fields in the dataset.

In [ ]:
# List available record sets by @id (Croissant 1.0+ stores record sets in metadata.record_sets)
record_sets = getattr(metadata, 'record_sets', [])
if not record_sets:
    # Try earlier field name
    record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    raise ValueError("No record sets found in the dataset metadata.")
print("Record sets and their @ids:")
for rs in record_sets:
    print(f"- {rs.id} (name={getattr(rs, 'name', 'N/A')})")

# For this dataset, select the main record set (there is typically only one for tabular data)
main_record_set = record_sets[0].id

# List fields for the main record set
fields = getattr(record_sets[0], 'fields', [])
if not fields:
    # Try earlier field name
    fields = getattr(record_sets[0], 'field', [])
print(f"\nFields in record set '{main_record_set}':")
for f in fields:
    col_name = getattr(f, 'name', 'N/A')
    data_type = getattr(f, 'data_type', getattr(f, 'dataType', 'N/A'))
    print(f"- @id: {f.id}, name: {col_name}, data_type: {data_type}")

## 3. Data Extraction
Load data from a specific record set into a Pandas DataFrame for analysis.
We use the record set and field `@id`s discovered above.

You can extract each record set by its `@id` and store in a Python dictionary.

In [ ]:
# Extract all record sets into dataframes
dataframes = {}
for rs in record_sets:
    print(f"Reading records from record set @id: {rs.id}")
    # Records generator returns one dict per row
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Shape: {df.shape}")
    print()
# Preview the columns and the first rows
print(f"Columns in record set '{main_record_set}':")
print(dataframes[main_record_set].columns.tolist())
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Let's run basic analysis, filtering, normalization, and grouping using only field `@id`s as column references.

We first select a numeric field, filter for significant values, normalize, and group by another categorical field.

In [ ]:
# Find a numeric field and a grouping field
import numpy as np

# For demonstration, look for likely numeric and categorical fields
# We'll search for 'abundance', 'peptide', 'coverage', or 'MW' for numeric; 'description' or 'accession' for group/cat
numeric_id = None
group_id = None
for f in fields:
    name = getattr(f, 'name', '').lower()
    dt = getattr(f, 'data_type', getattr(f, 'dataType', '')).lower()
    if (any(x in name for x in ['abundance', 'peptide', 'coverage', 'mw', 'count']) or dt in ['float', 'number', 'integer']):
        if numeric_id is None:
            numeric_id = f.id
    if (any(x in name for x in ['accession', 'description', 'protein']) or dt in ['text', 'string']):
        if group_id is None:
            group_id = f.id

# Fallback if not found
if not numeric_id:
    numeric_id = dataframes[main_record_set].select_dtypes(include=[np.number]).columns[0]
if not group_id:
    # Use first non-numeric column
    group_id = [c for c in dataframes[main_record_set].columns if dataframes[main_record_set][c].dtype == 'object'][0]

print(f"Selected numeric field: {numeric_id}")
print(f"Selected group field: {group_id}")

# Remove rows with missing values in the numeric column
df = dataframes[main_record_set].copy()
df[numeric_id] = pd.to_numeric(df[numeric_id], errors='coerce')
filtered_df = df[df[numeric_id] > 10].dropna(subset=[numeric_id])

print(f"Filtered records where {numeric_id} > 10 (showing top 5):")
print(filtered_df[[numeric_id, group_id]].head())

# Normalize numeric field
filtered_df[f"{numeric_id}_normalized"] = (filtered_df[numeric_id] - filtered_df[numeric_id].mean()) / filtered_df[numeric_id].std()
print(f"\nNormalized {numeric_id} (first 5):")
print(filtered_df[[numeric_id, f"{numeric_id}_normalized"]].head())

# Group by group_id and show the mean
if group_id in df.columns:
    grouped_df = filtered_df.groupby(group_id)[numeric_id].mean().reset_index()
    print(f"\nMean {numeric_id} by {group_id} (first 5):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships.

Let's plot the distribution of the selected numeric field (e.g., 'abundance') and a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.histplot(filtered_df[numeric_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_id}")
plt.xlabel(numeric_id)
plt.ylabel('Count')
plt.show()

# If the group is not too high in cardinality, plot a boxplot
if filtered_df[group_id].nunique() <= 20:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_id, y=numeric_id, data=filtered_df)
    plt.title(f"{numeric_id} by {group_id}")
    plt.xlabel(group_id)
    plt.ylabel(numeric_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook we demonstrated how to use the `mlcroissant` library to explore, extract, and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset schema for mass spectrometry analysis. 

- All data entities were referenced by their `@id`, ensuring reproducibility.
- We loaded record sets, explored fields, filtered and normalized data, performed grouping, and visualized numeric distributions.
- The workflow can be extended for further proteomic or mass-spec data analysis as required.

For additional data details and schema insights, consult the [full Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).